<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 1 &middot; Week 1 &middot; Part 2 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Model Representation

You now write your first machine-learning model. It is a straight line, and by the end of
this lab you will be able to state exactly what its two numbers mean and what they cost you
when they are wrong.

**Companion theory lecture:** CSE 815, Week 1, Session 1 (the linear regression model).

## What you will be able to do

1. Store a training set as NumPy arrays and read off $m$, $x^{(i)}$ and $y^{(i)}$.
2. Implement $f_{w,b}(x) = wx + b$ both with a loop and vectorised, and show they agree.
3. Plot a candidate model over the data and judge it by eye.
4. Use a fitted model to predict the target for an unseen input.

---

## 1. The training set

A supervised training set is a list of $m$ pairs

$$\mathcal{D} = \{(x^{(1)}, y^{(1)}), (x^{(2)}, y^{(2)}), \dots, (x^{(m)}, y^{(m)})\}.$$

We use flat prices from Chattogram: the input $x$ is floor area in thousands of square feet,
the target $y$ is price in lakh BDT.

> Remember the notation from the lecture: the parenthesised superscript $x^{(i)}$ is an
> **index**, not a power. In code that is just `x_train[i]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

# x = size in 1000 sq ft, y = price in lakh BDT
x_train = np.array([0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.4])
y_train = np.array([30., 36., 45., 48., 58., 62., 70., 82.])

print("x_train:", x_train)
print("y_train:", y_train)
print("shapes :", x_train.shape, y_train.shape)

In [ ]:
# m is the number of training examples. Two ways to get it -- both are used in the wild.
m = x_train.shape[0]
print("m via shape:", m)
print("m via len  :", len(x_train))

# A single training example.
i = 3
print(f"\n(x^({i}), y^({i})) = ({x_train[i]}, {y_train[i]})")

# Every example, printed with its index.
print()
for i in range(m):
    print(f"  i={i}:  x = {x_train[i]:.1f} k sq ft   ->   y = {y_train[i]:.0f} lakh")

### Always look at the data first

Before fitting anything, plot it. Thirty seconds here saves an hour of debugging a model
that was never going to work.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(x_train, y_train, marker="o", s=60, c="#1B7F79", label="Training data")
plt.title("Flat price against floor area")
plt.xlabel("Size (1000 sq ft)")
plt.ylabel("Price (lakh BDT)")
plt.legend()
plt.show()

The points fall close to a straight line that rises to the right. So a linear model is a
reasonable choice here &mdash; which is a claim about *this data set*, not a universal truth.
Part of your job as an engineer is to look and check.

---

## 2. The model

$$f_{w,b}(x) = wx + b$$

- $w$ is the **weight** (slope): lakh BDT per thousand square feet.
- $b$ is the **bias** (intercept): the predicted price of a flat of size zero.

$w$ and $b$ are the **parameters**. Training means choosing them. For now we choose them by
hand, so that you can see what the choice does.

In [ ]:
w = 30.0
b = 5.0
print(f"f(x) = {w}x + {b}")

### Version 1: with a loop

This mirrors the mathematics one symbol at a time. It is the version to write first when
you are unsure, and the version to check the fast one against.

In [ ]:
def compute_model_output_loop(x, w, b):
    """
    Compute the linear model prediction for every training example, using a loop.

    Args:
        x (ndarray (m,)): input features
        w, b (scalar)   : model parameters

    Returns:
        f_wb (ndarray (m,)): the model prediction for each example
    """
    m = x.shape[0]
    f_wb = np.zeros(m)              # allocate the output first
    for i in range(m):
        f_wb[i] = w * x[i] + b
    return f_wb


preds_loop = compute_model_output_loop(x_train, w, b)
print("predictions:", preds_loop)

### Version 2: vectorised

Exactly the same arithmetic, expressed once for the whole array. This is the version you
will actually use.

In [ ]:
def compute_model_output(x, w, b):
    """
    Compute the linear model prediction for every training example (vectorised).

    Args:
        x (ndarray (m,)): input features
        w, b (scalar)   : model parameters

    Returns:
        f_wb (ndarray (m,)): the model prediction for each example
    """
    return w * x + b


preds_vec = compute_model_output(x_train, w, b)
print("predictions:", preds_vec)

### Check they agree

Never compare floating-point arrays with `==`. Rounding makes two mathematically equal
results differ in the last bit. Use `np.allclose`, which asks whether they agree *to within
a tolerance*.

In [ ]:
print("equal with == :", np.all(preds_loop == preds_vec))
print("allclose      :", np.allclose(preds_loop, preds_vec))

assert np.allclose(preds_loop, preds_vec), "the two implementations disagree"
print("\nOK - the loop and the vectorised version agree.")

---

## 3. Seeing the model against the data

In [ ]:
def plot_model(w, b, ax=None, colour="#9B2C4B", show_errors=False):
    """Scatter the training data and draw the line f(x) = wx + b over it."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))

    f_wb = compute_model_output(x_train, w, b)

    if show_errors:
        for i in range(x_train.shape[0]):
            ax.plot([x_train[i], x_train[i]], [y_train[i], f_wb[i]],
                    c="#C97B17", lw=1.2, zorder=1)

    ax.scatter(x_train, y_train, marker="o", s=55, c="#1B7F79",
               label="Training data", zorder=3)
    ax.plot(x_train, f_wb, c=colour, lw=2,
            label=f"f(x) = {w:g}x + {b:g}", zorder=2)
    ax.set_xlabel("Size (1000 sq ft)")
    ax.set_ylabel("Price (lakh BDT)")
    ax.legend()
    return ax


ax = plot_model(w=30.0, b=5.0)
ax.set_title("A first guess")
plt.show()

### Changing the parameters

$b$ shifts the line up and down; $w$ tilts it. Run the next cell and watch which one moves.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

plot_model(w=20.0, b=5.0, ax=axes[0]); axes[0].set_title("w too small")
plot_model(w=33.0, b=4.0, ax=axes[1]); axes[1].set_title("about right")
plot_model(w=45.0, b=5.0, ax=axes[2]); axes[2].set_title("w too large")

for a in axes:
    a.set_ylim(0, 110)
plt.tight_layout()
plt.show()

### The errors are what we will minimise

The vertical orange bars below are the **residuals** $f_{w,b}(x^{(i)}) - y^{(i)}$. A good
model makes them short. Part 3 turns the total length of these bars into a single number.

In [ ]:
ax = plot_model(w=30.0, b=5.0, show_errors=True)
ax.set_title("Residuals of a mediocre model")
plt.show()

residuals = compute_model_output(x_train, 30.0, 5.0) - y_train
print("residuals:", np.round(residuals, 2))
print(f"largest error: {np.max(np.abs(residuals)):.2f} lakh")

### Checkpoint 1

Try to beat `w = 30, b = 5` by hand. Call `plot_model(w=..., b=..., show_errors=True)` with
a few of your own values and print the largest absolute residual each time.

Record the best `(w, b)` you find. Keep it &mdash; you will compare it with what gradient
descent finds in Part 4.

In [ ]:
# Your code here

---

## 4. Making a prediction

The point of fitting a model is to answer a question about an input you have never seen.

In [ ]:
w = 33.0
b = 4.0

x_new = 1.5                              # a 1500 sq ft flat
y_hat = compute_model_output(np.array([x_new]), w, b)[0]

print(f"A {x_new * 1000:.0f} sq ft flat is predicted at {y_hat:.1f} lakh BDT")

> **Interpolation versus extrapolation.** `x_new = 1.5` sits inside the range of the training
> data (0.8 to 2.4), so the model is interpolating and we can have some confidence. Ask it
> about a 10,000 sq ft flat and it will answer without hesitating &mdash; and the answer will be
> worthless, because no such example was ever seen. Run the next cell and look at the number.

In [ ]:
for x_new in [0.5, 1.5, 2.2, 10.0]:
    y_hat = w * x_new + b
    inside = x_train.min() <= x_new <= x_train.max()
    flag = "interpolating" if inside else "EXTRAPOLATING - do not trust"
    print(f"x = {x_new:5.1f}k sq ft  ->  {y_hat:7.1f} lakh   ({flag})")

### Checkpoint 2

Write a function `predict(x, w, b)` that accepts either a single number or a NumPy array and
returns the prediction(s). Then use it to predict the price of flats of 0.9, 1.3 and 2.1
thousand sq ft with `w = 33, b = 4`.

*Expected:* approximately `33.7`, `46.9`, `73.3`.

In [ ]:
# Your code here

---

## 5. Recap

- A training set is `x_train`, `y_train` and $m$; `x_train[i]` is $x^{(i)}$.
- The model $f_{w,b}(x) = wx + b$ is one line of vectorised NumPy: `w * x + b`.
- Write the loop version first, then the vectorised one, then `assert np.allclose(...)`.
- $w$ tilts the line, $b$ shifts it.
- The residual $f_{w,b}(x^{(i)}) - y^{(i)}$ is what the next lab will measure.
- A model outside the range of its training data is guessing.

### Exercises to hand in

1. Add a ninth training example, a 3.0k sq ft flat at 95 lakh. Re-plot. Does your
   hand-chosen `(w, b)` still look good? Explain what changed and why.
2. Implement `compute_model_output_matrix(X, w, b)` where `X` has shape `(m, 1)`, using
   `np.dot`. Confirm it agrees with the 1-D version via `np.allclose`. *(This is the shape
   you will need in Week 2 for multiple features.)*
3. For `w = 33, b = 4`, compute the residual vector and report its mean. It is close to
   zero even though individual errors are large. Explain why the **mean residual** is a
   poor measure of fit, and what the lecture proposed instead.
4. The model predicts a price for a flat of size 0 &mdash; namely $b$. Is `b = 4` lakh
   meaningful? Discuss in three or four sentences what the intercept means when $x = 0$
   lies outside the data.

### Next

**Part 3 &mdash; Cost Function Implementation:** turn those residuals into $J(w,b)$, and see
its shape.

**Reading:** James et al., *ISL* 2e, &sect;3.1.1.